# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [ ]:
!pip install --upgrade --quiet datasets fsspec==2025.3.0 huggingface_hub transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.6 MB/s eta 0:00:00


---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm

In [ ]:
# batch_size와 epochs를 조정해보세요!
batch_size = 16
epochs = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 데이터셋 로드
raw_datasets = load_dataset("nyu-mll/glue", "sst2")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [ ]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [ ]:
# 학습 함수 정의
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized_datasets = raw_datasets.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    train_dataset = tokenized_datasets["train"]
    valid_dataset = tokenized_datasets["validation"]

    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")
    return acc

# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")


======== Now Training: bert-base-uncased ========


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 4210/4210 [26:02<00:00,  2.70it/s]


Epoch 1 - Avg Train Loss: 0.1819


Epoch 2: 100%|██████████| 4210/4210 [26:01<00:00,  2.70it/s]


Epoch 2 - Avg Train Loss: 0.1150
Validation Accuracy (google/electra-base-discriminator): 0.9404


## 📊 결과 비교 및 분석
**1. 각 모델 구조 설명**

- **BERT**: Transformer encoder 기반. MLM(Masked Language Modeling) 방식으로 사전 학습 - 입력 토큰의 15%를 [MASK]로 가린 뒤 원래 토큰을 예측한다. 양방향 문맥을 동시에 처리하므로 NLU 태스크에 강하다.

- **ELECTRA**: Generator-Discriminator 구조. Generator가 일부 토큰을 그럴듯한 가짜 토큰으로 대체하면, Discriminator(ELECTRA 본체)가 각 토큰이 원본인지 대체된 것인지 이진 판별한다.(RTD: Replaced Token Detection). MLM이 전체 토큰의 15%만 학습 신호로 쓰는 것과 달리, RTD는 모든 토큰에 대해 학습 신호를 생성하므로 같은 연산량 대비 더 효율적이다.

**2. 어떤 모델이 적합한지에 대한 의견**
제한된 학습 자원(시간, GPU) 환경에서는 ELECTRA가 더 적합하다고 생각한다. 같은 에포크나 배치 조건에서 ELECTRA는 모든 토큰에 대해 학습 신호가 발생하므로 BERT보다 빠르게 수렴하며, SST-2 같은 분류 태스크에서 accuracy도 비슷하거나 소폭 높은 경향이 있다. 반면 BERT는 생태계가 넓고 파생 모델(RoBERTa, DeBERTa 등)이 많아 fine-tuning 레퍼런스가 풍부하다는 실용적 장점이 있으므로, 참고 자료 접근성이 중요한 상황에서는 BERT도 여전히 유효한 선택이다.

